# [LAB] Harness 실습: SKILL.md로 Agent 동작 바꾸기

Agent가 읽을 스킬 문서(SKILL.md)를 쓰고, 그 문서를 읽고 따르는 Agent를 만들어 실행합니다. 그다음 코드와 모델은 그대로 두고 값 하나만 바꿔 Agent의 동작이 달라지는 것을 봅니다.

- **2장 Agent가 읽을 스킬 쓰기**: 삼성SDS 보고서 양식이 적힌 SKILL.md를 파일로 씁니다.
- **3장 스킬을 읽는 Agent 만들기**: 모델·도구·작업 공간·스킬 폴더·시스템 프롬프트를 묶어 Agent를 만들고, 보고서 요청과 범위 밖 요청을 차례로 실행합니다. 시스템 프롬프트를 지워 범위 밖 요청에 어떻게 답하는지도 봅니다.
- **4장 SKILL.md 고쳐 동작 바꾸기**: SKILL.md의 스킬 설명 한 줄을 바꾸고 Agent를 다시 실행해 결과를 비교합니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다. Agent를 실행하는 셀은 모델을 여러 번 호출하므로 결과가 나오기까지 수십 초가 걸리고, 답은 실행할 때마다 조금씩 다릅니다.

In [ ]:
%cd ../../../course/Chapter02/notebooks

**목차**

- 1. 실습 방법
- 2. Agent가 읽을 스킬 쓰기
  - 2.1 SKILL.md 쓰기
- 3. 스킬을 읽는 Agent 만들기
  - 3.1 검색 도구
  - 3.2 Agent 만들기
  - 3.3 두 요청 실행하기
- 4. SKILL.md 고쳐 동작 바꾸기
  - 4.1 스킬 설명 바꿔 다시 실행하기
- 5. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env()

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고칩니다. `.env`는 노트북을 시작할 때 한 번만 읽으므로, 고친 뒤에는 커널을 다시 시작하고 맨 위 셀부터 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

도구를 부를 때 `Error in ConsoleCallbackHandler.on_tool_start callback: KeyError('input')` 같은 줄이 섞여 보일 수 있습니다. 디버그 내용을 출력하는 쪽의 문제라 실행 결과에는 영향이 없으니 그대로 두면 됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

<details><summary>강사용 메모</summary>

- 원본: `course/Chapter02/code/raw/04-harness/agent.py`, `skills/samsung-sds-report/SKILL.md`. 수강생은 .py 파일을 보지 않습니다. SKILL.md는 `%%writefile`로 노트북 폴더 아래 `harness/skills/samsung-sds-report/SKILL.md`에 씁니다. `harness/`는 Git 제외 대상입니다(.gitignore).
- Agent의 작업 공간(`FilesystemBackend(root_dir=...)`)을 노트북 폴더가 아니라 `harness/`로 좁혔습니다(patch `HARNESS_DIR = Path("harness")`, 목차 2.0 A안). 원본은 `Path(__file__).parent`입니다. 슬라이드 코드는 `root_dir="."`입니다.
- 원본의 `skills.md`(역할·행동 규칙 문서)는 코드가 읽지 않아 넣지 않았습니다(2026-09-28 사용자 결정 7). solution에만 있는 `ai_agent_trends_report.md`도 넣지 않았습니다.
- 사외 실측(2026-09-28, `COURSE_ENV=out`, `gpt-6-luna`): 사례 1(보고서) 약 17~20초, LLM 약 3~5회(SKILL.md 읽기 1회 + 검색 2~4회 + 작성). 사례 2(퀵소트) 약 2초, 도구 호출 없이 거절. 사내 게이트웨이의 도구 호출(deepagents는 모든 단계가 도구 호출)은 **미확인**입니다. 안 되면 이 노트북 전체가 멈춥니다. 강의 전 1순위 확인 항목입니다.
- 원본 `chat()`은 `response = ''`를 채우지 않아 늘 빈 문자열을 돌려줍니다. 반환값을 쓰는 곳이 없어 화면에는 영향이 없어 그대로 두었습니다.

</details>

## 1. 실습 방법

코드는 모두 완성되어 있습니다. 위에서부터 차례로 실행합니다.

- `# ✍️ CHANGE_HERE:` 주석이 있는 셀은 먼저 그대로 실행해 결과를 봅니다. 그다음 셀 위의 **바꿔 볼 값**을 복사해, 주석 바로 아래 줄의 표시된 자리(지금 값)를 지우고 붙여 넣습니다. 어떻게 달라질지 생각해 본 뒤 다시 실행합니다.
- 달라진 결과를 본 뒤 셀 아래 접힌 **무슨 일이 생기나**를 펼쳐 비교합니다. 확인했으면 값을 원래대로 되돌립니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. Agent가 읽을 스킬 쓰기

SKILL.md는 Agent가 특정 작업을 할 때 참고하는 절차 문서입니다. 이 실습의 스킬은 "삼성SDS 보고서 양식으로 보고서 쓰기"입니다. 문서는 두 부분입니다.

- **맨 위 `---` 사이**: 스킬 이름(`name`)과 설명(`description`). 설명에는 이 스킬을 **언제** 쓰는지 적습니다.
- **그 아래 본문**: 보고서 양식과 작성 규칙. 이 스킬을 **어떻게** 쓰는지 적습니다.

아래 셀을 실행하면 이 폴더 아래 `harness/skills/samsung-sds-report/SKILL.md` 파일이 만들어집니다. 다음 장에서 만들 Agent가 이 파일을 읽습니다. 모델은 호출하지 않습니다.

In [ ]:
from pathlib import Path

Path("harness/skills/samsung-sds-report").mkdir(parents=True, exist_ok=True)

<details><summary>강사용 메모</summary>

- 원본 SKILL.md(raw·solution 동일)를 그대로 씁니다. 빌더가 `%%writefile` 셀 앞에 폴더를 만드는 숨은 셀을 넣습니다.
- 양식 속 "외부에 공개되지 않으므로 이 스킬 파일을 읽어야만 올바른 형식으로 작성할 수 있다"는 설명은 실습용 설정입니다. 모델이 이 파일을 읽지 않으면 SDS 양식을 알 수 없다는 점을 보여 주려는 장치입니다.

</details>

### 2.1 SKILL.md 쓰기

In [ ]:
%%writefile harness/skills/samsung-sds-report/SKILL.md
---
name: samsung-sds-report
description: 삼성SDS AI연구소 공식 보고서 작성 스킬. 사용자가 "보고서", "리포트", "SDS 양식" 등을 언급하며 작성을 요청할 때 반드시 발동한다. 삼성SDS 내부 보고서 양식(SDS-AI-RPT 포맷)은 외부에 공개되지 않으므로 이 스킬 파일을 읽어야만 올바른 형식으로 작성할 수 있다.
---

# samsung-sds-report

## 삼성SDS AI연구소 보고서 작성 지침

이 스킬이 발동되면 아래 SDS-AI-RPT 양식을 반드시 사용합니다.
이 양식은 삼성SDS 내부 표준으로, 다른 형식으로 작성하면 안 됩니다.

---

## SDS-AI-RPT 보고서 양식

보고서는 아래 템플릿을 그대로 따릅니다. 섹션 코드([SDS-...])는 생략하지 않습니다.

```
[SDS-AI-RPT v2.1]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
보안등급: INTERNAL  |  배포범위: AI연구소 내부
작성일: YYYY-MM-DD  |  문서번호: AI-RPT-YYYYMM-NNN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

■ TL;DR (3줄 요약)
① [핵심 발견 1]
② [핵심 발견 2]
③ [권고 사항]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[SDS-1.0] 개요
  1.1 작성 배경
  [배경 설명]
  
  1.2 조사 범위
  [범위 설명]

[SDS-2.0] 현황 분석
  2.1 핵심 동향
  | 구분 | 내용 | SDS 영향도 |
  |------|------|-----------|
  | [항목] | [내용] | 상/중/하 |

  2.2 주요 플레이어
  [내용]

[SDS-3.0] 결론 및 제언
  3.1 결론
  [결론]
  
  3.2 SDS 대응 방향 (권고)
  - [권고 1]
  - [권고 2]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[SDS-END] 개정 이력
| 버전 | 일자 | 변경 내용 |
|------|------|---------|
| v1.0 | YYYY-MM-DD | 최초 작성 |
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```

## 작성 규칙

1. `[SDS-AI-RPT v2.1]` 헤더로 시작합니다.
2. `■ TL;DR` 섹션은 반드시 3줄로 작성합니다.
3. 섹션 코드 `[SDS-1.0]`, `[SDS-2.0]`, `[SDS-3.0]` 은 생략하지 않습니다.
4. `[SDS-2.0]` 현황 분석에는 반드시 "SDS 영향도(상/중/하)" 컬럼이 있는 표를 포함합니다.
5. `[SDS-END]` 개정 이력 표로 마무리합니다.

위 셀을 실행하면 Agent가 읽을 SKILL.md 파일이 만들어집니다. 이 장에서는 파일만 씁니다. 다음 장에서 Agent를 만들어 실행합니다.

**볼 지점**
- 설명(`description`)에 "보고서", "리포트", "SDS 양식"을 언급하며 작성을 요청할 때 쓰라고 적혀 있습니다. Agent는 처음에 이 이름과 설명만 보고, 필요하다고 판단하면 파일 전체를 읽습니다.
- 본문의 양식은 `[SDS-AI-RPT v2.1]` 헤더로 시작합니다. 다음 장에서 Agent의 답이 이 헤더로 시작하는지 보면, Agent가 스킬을 따랐는지 알 수 있습니다.
- 작성 규칙 다섯 줄 중 다음 장 출력에서 확인할 수 있는 것은 첫 줄(헤더)입니다. 출력은 앞 200자까지만 보이기 때문입니다.

## 3. 스킬을 읽는 Agent 만들기

`create_deep_agent`로 Agent를 만듭니다. 이 함수는 Agent가 일하는 환경을 한 번에 묶습니다.

- **모델**과 **도구**: 검색 도구 `search_web`. 실제로 검색하지 않고 정해 둔 글(검색 결과 흉내)을 돌려줍니다.
- **작업 공간** `FilesystemBackend`: Agent가 파일을 읽고 쓸 수 있는 폴더입니다. 이 실습에서는 `harness` 폴더만 Agent에게 보여 줍니다. Agent에게는 이 폴더가 맨 위(`/`)로 보여서, SKILL.md는 `/skills/samsung-sds-report/SKILL.md`로 보입니다.
- **스킬 폴더** `skills`: 스킬 폴더들을 담은 상위 폴더(`skills/`)입니다. Agent는 그 아래 폴더마다 든 SKILL.md를 찾습니다.
- **시스템 프롬프트**: 범위를 벗어난 요청을 거절하게 합니다. 스킬은 보고서 요청을 어떻게 처리할지만 알려 주고, 보고서와 관계없는 요청을 막지는 않기 때문입니다.
- **체크포인터** `MemorySaver`: 대화 상태를 기억합니다.

마지막 셀은 두 요청을 차례로 보냅니다. 사례 1은 삼성SDS 양식 보고서 요청, 사례 2는 범위를 벗어난 퀵소트 코드 요청입니다. 출력은 Agent가 한 단계를 마칠 때마다 한 줄씩 나옵니다. 줄 앞의 `[HumanMessage]`는 사람의 요청, `[AIMessage]`는 모델의 답, `[ToolMessage]`는 도구 실행 결과입니다.

**왜 배우나:** 하네스는 모델에게 더 긴 지시문을 주는 것이 아니라, 모델이 일할 환경을 만드는 일입니다. 이 셀 하나에 모델·도구·작업 공간·규칙·기억이 모두 들어갑니다.

준비 코드입니다. 필요한 도구를 불러오고, Agent의 작업 폴더를 `harness`로 정합니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
"""
Harness Engineering 실습

deepagents의 create_deep_agent + skills 디렉토리로 Agent를 빌드하는 예제.

skills/samsung-sds-report/SKILL.md 가 Agent의 행동 규칙을 정의한다.
Agent는 description을 보고 이 스킬이 필요한지 판단한 뒤 SKILL.md를 읽어 지시사항을 따른다.

→ SKILL.md만 수정하면 모델·코드 변경 없이 Agent 동작을 조정할 수 있다.
"""

from pathlib import Path

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver

from config.llm import get_llm

load_dotenv()

HARNESS_DIR = Path("harness")  # Agent의 작업 폴더. SKILL.md가 이 폴더 아래에 있습니다

<details><summary>강사용 메모</summary>

- 노트북에서만 원본을 손봤습니다(patch, 원본 파일은 그대로): 준비 코드의 `sys.stdout.reconfigure(encoding="utf-8")`와 `import sys` 삭제(Jupyter 출력 객체에는 `reconfigure`가 없어 `AttributeError`), `HARNESS_DIR = Path("harness")`, docstring의 틀린 스킬 경로(`skills/research-writing` → `samsung-sds-report`), 출력 길이 `content[:80]` → `[:200]`(solution 쪽. 80자면 SKILL.md를 읽는 줄과 보고서 머리가 잘 안 보입니다), TODO 둘째 줄 오타("프롬프트룰"), `skills` 설명 주석(원본 "SKILL.md 파일이 들어있는 디렉토리 목록"은 deepagents 동작과 반대라 "스킬 폴더들을 담은 상위 디렉토리"로. 2026-09-29 리뷰).
- 빈칸은 `agent = create_deep_agent(...)` 문장 전체입니다. 목차는 인자 줄만 빈칸으로 두려 했지만, 빌더의 빈칸 끝 표시가 `checkpointer=MemorySaver(),` 줄의 괄호와 구분되지 않아 문장 전체로 잡았습니다. TODO 두 줄은 patch로 문장 위로 옮겼습니다.
- 정답 보기는 solution 쪽 주석·프롬프트입니다("보고서·리서치 범위를 벗어난 요청"). raw의 이전 강사 말투 주석 3줄, solution의 `MCP_CONFIG`·`await` 주석과 `# middleware`는 쓰이지 않는 코드라 뺐습니다. `recursion_limit`은 raw의 40입니다.
- `FilesystemBackend(virtual_mode=True)`는 `root_dir` 밖 경로를 막지만 샌드박스는 아닙니다. deepagents는 `ls`·`read_file`·`write_file`·`edit_file`·`glob`·`grep` 도구를 기본으로 붙입니다. 사외 실측에서는 보고서를 파일로 저장하지 않았습니다.

</details>

### 3.1 검색 도구

In [ ]:
# ── 도구 정의 ─────────────────────────────────────────────────────────────────


@tool
def search_web(query: str) -> str:
    """웹에서 주제를 검색한다."""
    # 실제 구현에서는 Tavily, SerpAPI 등을 사용한다.
    return (
        f"[검색: {query}]\n"
        f"- 최근 동향: 빠른 성장세, 기업 도입 증가\n"
        f"- 핵심 기술: 트랜스포머 아키텍처, RLHF, RAG\n"
        f"- 주요 사례: 고객 지원 자동화, 코드 리뷰, 문서 요약\n"
        f"- 시장 규모: 2024년 기준 약 2,000억 달러 (연 30% 성장 예상)"
    )

### 3.2 Agent 만들기

**✍️ CHANGE_HERE: 시스템 프롬프트를 더 분명한 거절 지시로 바꾸면 사례 2(퀵소트 코드 요청)에 어떻게 답할까요?**

**바꿔 볼 값** — 바로 아래 줄의 `코드 작성, 수학 계산 등 보고서·리서치 범위를 벗어난 요청은 정중히 거절한다.` 자리에 복사해 붙여 넣기

따옴표 안의 문장 전체를 지우고 붙여 넣기
```text
너는 보고서·리서치 전용 Agent다. 코드 작성, 수학 계산 등 범위를 벗어난 요청에는 코드나 계산 결과 없이 한 문장으로 정중히 거절한다.
```

이 셀과 "두 요청 실행하기" 셀을 차례로 다시 실행해 사례 2의 마지막 `[AIMessage]`를 바꾸기 전과 비교합니다. 확인했으면 원래 문장으로 되돌리고 이 셀을 다시 실행합니다.

`create_deep_agent`가 Agent가 일하는 환경을 한 번에 묶습니다. 모델(`get_llm()`), 도구(`search_web`), 작업 공간(`backend`), 스킬 폴더(`skills`), 시스템 프롬프트(`system_prompt`), 체크포인터(`MemorySaver()`)입니다. `skills`에는 SKILL.md가 바로 든 폴더가 아니라 스킬 폴더들을 담은 상위 폴더(`"skills/"`)를 적습니다. 이 셀은 Agent만 만들고 아무것도 출력하지 않습니다. 결과는 아래 "두 요청 실행하기" 셀에서 봅니다.

In [ ]:
# ── Agent 생성 ────────────────────────────────────────────────────────────────
# FilesystemBackend: 로컬 파일시스템을 Agent의 작업 공간으로 사용
# skills: 스킬 폴더들을 담은 상위 디렉토리 목록 (backend root_dir 기준 상대경로). 그 아래 폴더마다 SKILL.md가 있다
#   → Agent는 SKILL.md의 name+description을 보고 "이 스킬이 필요하다" 판단 후 read_file로 상세 지시 로드

# virtual_mode=True: 에이전트에게 가상 경로(/skills/... 등)로 파일시스템을 노출
# ls 결과가 Windows 절대경로 대신 /skills/ 같은 가상경로로 반환되어 경로 혼동 없음
backend = FilesystemBackend(root_dir=str(HARNESS_DIR), virtual_mode=True)

agent = create_deep_agent(
    model=get_llm(),
    tools=[search_web],
    backend=backend,
    skills=["skills/"],
    # ✍️ CHANGE_HERE: 시스템 프롬프트를 더 분명한 거절 지시로 바꾸면 사례 2(퀵소트 코드 요청)에 어떻게 답할까요?
    system_prompt="코드 작성, 수학 계산 등 보고서·리서치 범위를 벗어난 요청은 정중히 거절한다.",
    checkpointer=MemorySaver(),
)

<details><summary>무슨 일이 생기나</summary>

바꾸기 전(지금 값)에는 이번 사외 모델이 사례 2에 `퀵소트를 구현하는 파이썬 코드는 다음과 같습니다:` 뒤에 `def quicksort(arr):` 코드를 그대로 써 주었습니다(2번 모두). "정중히 거절한다" 한 줄로는 모델이 거절하지 않은 것입니다. 바꾼 값으로는 사례 2가 `코드 작성 요청은 처리할 수 없습니다. 양해 부탁드립니다.` 한 문장으로 끝났습니다(2번 모두, 도구 호출 없음). 사례 1(보고서)은 바꾼 뒤에도 SKILL.md를 읽고 보고서를 썼습니다(1번 확인).

거절은 스킬이 아니라 시스템 프롬프트가 정합니다. 무엇을 거절할지와 어떻게 거절할지(코드 없이 한 문장)를 분명히 적어야 모델이 지킵니다. 모델마다 따르는 정도가 달라서, 강의장 모델은 지금 값으로도 거절할 수 있습니다.

</details>

### 3.3 두 요청 실행하기

In [ ]:
# ── 대화형 실행 ───────────────────────────────────────────────────────────────


def chat(session_id: str, message: str) -> str:
    # recursion_limit: LangGraph 노드 실행 횟수 상한 — 기본값 25, 무한루프 방지
    config = {"configurable": {"thread_id": session_id}, "recursion_limit": 40}
    response = ""
    for chunk in agent.stream(
        {"messages": [HumanMessage(content=message)]},
        config=config,
        stream_mode="values",
    ):
        last_msg = chunk["messages"][-1]
        msg_type = type(last_msg).__name__
        content = last_msg.content or ""
        # 각 단계마다 메시지 타입과 내용 앞부분을 출력해 어디서 멈추는지 확인
        print(f"[{msg_type}] {content[:200]}", flush=True)
    print()
    return response


print("=== Harness Agent (skills/ 디렉토리 기반) ===")
print(f"스킬 경로: {HARNESS_DIR / 'skills'}\n")

# 케이스 1: SDS 보고서 양식을 명시 → 모델이 SKILL.md를 읽어야만 형식을 알 수 있음
# 케이스 2: 범위 이탈 → system_prompt로 거절
test_cases = [
    ("session_1", "AI Agent 최신 트렌드를 삼성SDS 보고서 양식으로 작성해줘."),
    ("session_2", "파이썬으로 퀵소트 코드를 짜줘."),
]

for session_id, message in test_cases:
    print(f"사용자: {message}")
    print("Agent: ", end="", flush=True)
    chat(session_id, message)
    print()

**볼 지점**
- 사례 1의 흐름: `[AIMessage]` 줄이 비어 있으면 모델이 글 대신 도구 호출을 보낸 단계입니다. 그 아래 `[ToolMessage]`가 도구 실행 결과입니다. 먼저 SKILL.md를 읽고(`@@ lines 1-…` 아래에 파일 내용), 검색을 몇 번 한 뒤, 마지막 `[AIMessage]`에서 보고서를 씁니다. 각 줄은 앞 200자까지만 출력합니다.
- 사례마다 `[HumanMessage]` 줄이 두 번 나옵니다. 처음 받은 요청을 한 번, 스킬 목록을 읽는 준비 단계를 마친 뒤 한 번 더 출력한 것입니다. 요청이 두 번 간 것은 아닙니다.
- 사례 2는 시스템 프롬프트에 따라 도구 없이 거절해야 합니다. 모델에 따라 거절하지 않고 코드를 써 주기도 합니다. 그때는 3.2의 CHANGE_HERE 값으로 바꿔 봅니다.
- 검색 결과는 정해 둔 글이라 늘 같습니다. 검색어와 검색 횟수는 실행할 때마다 달라집니다.

<details><summary>강사용 메모</summary>

- 사외 실측에서 `[ToolMessage]`의 SKILL.md 줄은 `@@ lines 1-71 of 71 @@` 다음 줄부터 파일 내용(`---`, `name: …`)으로 나왔습니다(deepagents 0.7.18 `read_file` 출력).
- `[HumanMessage]` 줄이 사례마다 두 번 나옵니다(스트림 첫 상태와 SkillsMiddleware `before_agent`가 `skills_metadata`를 채운 뒤 상태. 2026-09-29 가짜 모델로 확인). 볼 지점에 한 줄로 적었습니다.
- 사외 실측에서 검색어를 영어로 만들고, 도구가 준 적 없는 기관 이름(Gartner·McKinsey)을 검색어에 넣기도 했습니다. 보고서 본문에 출처를 지어내는지는 4장에서 짚을 수 있습니다.
- 사외 재실측(2026-10-21, OpenRouter `qwen/qwen3-235b-a22b-2507`, OpenAI 크레딧 소진으로 대체): 원래 시스템 프롬프트로 사례 2에 퀵소트 코드를 써 줌(2회 모두, 노트북 전체 실행 1회 포함). 시스템 프롬프트를 지워도 코드(2회). 바꿔 볼 값(분명한 거절 지시)으로는 2회 모두 한 문장 거절. 사례 1은 SKILL.md를 읽고 `[SDS-AI-RPT v2.1]` 양식으로 썼으나, 마지막 답 앞 200자에 헤더 대신 "양식에 따라 작성했다"는 안내가 오기도 함(파일로 저장하는 경우). 사내 모델은 미확인.

</details>

## 4. SKILL.md 고쳐 동작 바꾸기

코드와 모델은 그대로 두고 SKILL.md만 바꿔 Agent의 동작이 달라지는지 봅니다. 아래 셀은 2장의 SKILL.md와 같고, `# ✍️ CHANGE_HERE:` 표시만 더했습니다. 맨 위 `---` 사이에서는 `#`으로 시작하는 줄이 주석이라 Agent가 읽는 내용에 영향이 없습니다.

한 번 바꿔 볼 때마다 세 셀을 차례로 실행합니다.

1. 아래 셀을 실행해 SKILL.md를 다시 씁니다.
2. 3장의 **Agent 만들기** 셀을 다시 실행합니다. Agent는 대화를 시작할 때 스킬 설명을 한 번 읽어 기억해 두므로, Agent를 새로 만들어야 바뀐 설명을 읽습니다.
3. 3장의 **두 요청 실행하기** 셀을 다시 실행하고, 바꾸기 전 결과와 비교합니다.

**왜 배우나:** 스킬 설명은 Agent가 이 스킬을 쓸지 판단하는 근거입니다. 코드를 고치지 않고 문서만 고쳐도 Agent의 동작이 바뀝니다.

In [ ]:
from pathlib import Path

Path("harness/skills/samsung-sds-report").mkdir(parents=True, exist_ok=True)

<details><summary>강사용 메모</summary>

- 스킬 캐시: deepagents 0.7.18의 SkillsMiddleware는 스킬 목록(name·description)을 스레드마다 한 번 읽어 상태에 저장합니다. 같은 체크포인터·같은 `session_1`로 다시 부르면 고친 설명이 반영되지 않습니다. 그래서 Agent 만들기 셀(새 `MemorySaver`)부터 다시 실행하게 했습니다.
- CHANGE_HERE는 frontmatter의 `description:` 줄입니다(2026-09-28 사용자 결정 7). YAML 주석이라 표시 줄이 문서를 해치지 않습니다. 설명에 콜론(`:`)을 넣으면 YAML 오류로 스킬이 조용히 빠집니다(경고 로그만 나옴).
- 사외 실측(2026-09-28, `gpt-6-luna`, 각 2회): 설명을 "회의록 요약 스킬. 사용자가 회의록을 요약해 달라고 할 때 발동한다."나 "영문 보고서에만 쓴다"로 바꿔도 Agent는 SKILL.md를 읽고 SDS 양식으로 썼습니다(요청에 "삼성SDS 보고서 양식"이 있고 스킬 이름이 `samsung-sds-report`라서로 보입니다. 추정). 당시 예시 "회의록 요약 스킬. 보고서 작성 요청에는 쓰지 않는다."는 2회 모두 SDS 양식을 쓰지 않았습니다.
- 사외 재실측(2026-10-19, `gpt-6-luna`): 위 예시 "회의록 요약 스킬. 보고서 작성 요청에는 쓰지 않는다."는 7회 모두 SKILL.md를 읽고 SDS 양식으로 썼습니다(노트북 4회·단독 3회). "보고서 작성에는 쓰지 않는다"만으로는 부족하고, "쓰는 때"를 좁혀 적어야 했습니다. 지금 예시 "회의록 요약 전용 스킬. 회의록 요약 요청이 아니면 쓰지 않는다. 보고서나 SDS 양식 요청에는 쓰지 않는다."는 14회 중 13회 SDS 양식을 쓰지 않았고(대개 SKILL.md를 읽지 않고 일반 보고서 형식으로 쓰거나 그 양식은 제공할 수 없다고 답함) 1회는 읽고 SDS 양식으로 썼습니다. "보고서나 SDS 양식 요청에는 절대 쓰지 않는다"나 "이 스킬을 읽지 않는다"만 적은 문구는 각 3회 모두 SDS 양식으로 썼습니다. 그래서 성공 기준은 "마지막 답이 SDS 양식이 아니다"로 적었습니다. 사내 모델의 반응은 미확인입니다.
- 사외 재실측(2026-10-21, OpenRouter `qwen/qwen3-235b-a22b-2507`): 이전 예시("회의록 요약 전용 스킬. 회의록 요약 요청이 아니면 쓰지 않는다. 보고서나 SDS 양식 요청에는 쓰지 않는다.")는 2회 모두 SKILL.md를 읽음. 지금 값("…보고서·리포트·SDS 양식 작성 요청에는 이 파일을 읽지 말고 일반 형식으로 답한다.")은 4회 중 2회 읽지 않고 일반 형식, 2회는 읽고 SDS 양식. "(사용 중지) 이 스킬은 더 이상 쓰지 않는다. 어떤 요청에도 이 파일을 읽지 않는다."는 2회 모두 읽지 않았지만 보고서 작성 자체를 거절해, "…일반 형식으로 쓴다"를 붙인 문구는 1회 읽고 일반 형식이라 쓰지 않음. 결과가 흔들리는 장이라 두 번쯤 실행하게 했습니다.

</details>

### 4.1 스킬 설명 바꿔 다시 실행하기

**✍️ CHANGE_HERE: 스킬 설명을 '보고서 요청에는 읽지 않는' 회의록 요약 스킬로 바꾸면 사례 1의 보고서는 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `삼성SDS AI연구소 공식 보고서 작성 스킬. 사용자가 "보고서", "리포트", "SDS 양식" 등을 언급하며 작성을 요청할 때 반드시 발동한다. 삼성SDS 내부 보고서 양식(SDS-AI-RPT 포맷)은 외부에 공개되지 않으므로 이 스킬 파일을 읽어야만 올바른 형식으로 작성할 수 있다.` 자리에 복사해 붙여 넣기

description: 뒤의 글 전체를 지우고 붙여 넣기. 콜론(:)은 넣지 않습니다
```text
회의록 요약 전용 스킬. 보고서·리포트·SDS 양식 작성 요청에는 이 파일을 읽지 말고 일반 형식으로 답한다.
```

이 셀을 실행해 SKILL.md를 다시 쓴 뒤, 3장의 **Agent 만들기** 셀과 **두 요청 실행하기** 셀을 차례로 다시 실행합니다. 결과가 실행마다 달라질 수 있으니 두 번쯤 실행해 봅니다. 확인했으면 원래 설명으로 되돌려 이 셀을 다시 실행하고, Agent 만들기 셀부터 다시 실행합니다.

In [ ]:
%%writefile harness/skills/samsung-sds-report/SKILL.md
---
name: samsung-sds-report
# ✍️ CHANGE_HERE: 스킬 설명을 '보고서 요청에는 읽지 않는' 회의록 요약 스킬로 바꾸면 사례 1의 보고서는 어떻게 될까요?
description: 삼성SDS AI연구소 공식 보고서 작성 스킬. 사용자가 "보고서", "리포트", "SDS 양식" 등을 언급하며 작성을 요청할 때 반드시 발동한다. 삼성SDS 내부 보고서 양식(SDS-AI-RPT 포맷)은 외부에 공개되지 않으므로 이 스킬 파일을 읽어야만 올바른 형식으로 작성할 수 있다.
---

# samsung-sds-report

## 삼성SDS AI연구소 보고서 작성 지침

이 스킬이 발동되면 아래 SDS-AI-RPT 양식을 반드시 사용합니다.
이 양식은 삼성SDS 내부 표준으로, 다른 형식으로 작성하면 안 됩니다.

---

## SDS-AI-RPT 보고서 양식

보고서는 아래 템플릿을 그대로 따릅니다. 섹션 코드([SDS-...])는 생략하지 않습니다.

```
[SDS-AI-RPT v2.1]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
보안등급: INTERNAL  |  배포범위: AI연구소 내부
작성일: YYYY-MM-DD  |  문서번호: AI-RPT-YYYYMM-NNN
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

■ TL;DR (3줄 요약)
① [핵심 발견 1]
② [핵심 발견 2]
③ [권고 사항]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[SDS-1.0] 개요
  1.1 작성 배경
  [배경 설명]
  
  1.2 조사 범위
  [범위 설명]

[SDS-2.0] 현황 분석
  2.1 핵심 동향
  | 구분 | 내용 | SDS 영향도 |
  |------|------|-----------|
  | [항목] | [내용] | 상/중/하 |

  2.2 주요 플레이어
  [내용]

[SDS-3.0] 결론 및 제언
  3.1 결론
  [결론]
  
  3.2 SDS 대응 방향 (권고)
  - [권고 1]
  - [권고 2]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[SDS-END] 개정 이력
| 버전 | 일자 | 변경 내용 |
|------|------|---------|
| v1.0 | YYYY-MM-DD | 최초 작성 |
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```

## 작성 규칙

1. `[SDS-AI-RPT v2.1]` 헤더로 시작합니다.
2. `■ TL;DR` 섹션은 반드시 3줄로 작성합니다.
3. 섹션 코드 `[SDS-1.0]`, `[SDS-2.0]`, `[SDS-3.0]` 은 생략하지 않습니다.
4. `[SDS-2.0]` 현황 분석에는 반드시 "SDS 영향도(상/중/하)" 컬럼이 있는 표를 포함합니다.
5. `[SDS-END]` 개정 이력 표로 마무리합니다.

<details><summary>무슨 일이 생기나</summary>

사례 1의 `[ToolMessage]`에서 SKILL.md 내용(`@@ lines 1-…`)이 사라지고, 마지막 답이 `### **AI Agent 최신 트렌드 보고서**`처럼 일반 보고서 형식으로 바뀌기도 합니다(실행마다 다름). 이번 사외 모델로 4번 실행해 2번은 SKILL.md를 읽지 않고 일반 형식으로 썼고("삼성SDS 보고서 양식은 내부 문서이므로 확인할 수 없어 일반 형식으로"라고 답하기도 함), 2번은 그래도 SKILL.md를 읽고 `[SDS-AI-RPT v2.1]` 양식으로 썼습니다.

Agent는 처음에 스킬의 이름과 설명만 보고 파일 전체를 읽을지 정합니다. 그래서 설명 한 줄로 동작이 바뀝니다. 다만 요청에 "삼성SDS 보고서 양식"이 직접 들어 있고 스킬 이름도 `samsung-sds-report`라, 모델이 설명보다 요청과 이름을 보고 읽기도 합니다. 읽었는지는 답의 모양보다 `[ToolMessage]`의 `@@ lines` 줄로 확인합니다.

</details>

SKILL.md를 다시 썼으면 3장으로 올라가 Agent 만들기 셀과 두 요청 실행하기 셀을 차례로 다시 실행합니다.

**볼 지점**
- 바꾼 것은 SKILL.md의 설명 한 줄뿐입니다. 모델도, Agent를 만드는 코드도 그대로입니다.
- 원래대로 되돌리려면 2장의 SKILL.md 쓰기 셀을 다시 실행하고, Agent 만들기 셀부터 다시 실행합니다.

## 5. 정리

**이 노트북에서 한 것**
- Agent가 읽을 스킬 문서(SKILL.md)를 파일로 썼습니다. 맨 위 설명은 스킬을 **언제** 쓰는지, 본문은 **어떻게** 쓰는지 알려 줍니다.
- `create_deep_agent`로 모델·도구·작업 공간·스킬 폴더·시스템 프롬프트·체크포인터를 묶어 Agent를 만들었습니다.
- 보고서 요청에는 Agent가 SKILL.md를 읽고 SDS 양식으로 썼고, 범위 밖 요청은 시스템 프롬프트에 따라 거절했습니다. 시스템 프롬프트를 지우면 범위 밖 요청에도 답했습니다.
- 코드와 모델은 그대로 두고 SKILL.md 설명 한 줄만 바꿔 Agent의 동작이 달라지는 것을 보았습니다.